In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import math
%matplotlib inline

In [2]:
iptu_2026 = pd.read_csv('Data/iptu/IPTU_2026.csv', sep=';')

In [3]:
iptu_2026.head()

,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NUMERO DA NL,DATA DO CADASTRAMENTO,NUMERO DO CONDOMINIO,CODLOG DO IMOVEL,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,COMPLEMENTO DO IMOVEL,BAIRRO DO IMOVEL,...,ANO DA CONSTRUCAO CORRIGIDO,QUANTIDADE DE PAVIMENTOS,TESTADA PARA CALCULO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TIPO DE TERRENO,FATOR DE OBSOLESCENCIA,ANO DE INICIO DA VIDA DO CONTRIBUINTE,MES DE INICIO DA VIDA DO CONTRIBUINTE,FASE DO CONTRIBUINTE
0,0010030001-4,2026,1,01/01/26,00-0,03812-1,R S CAETANO,13.0,NaN,SANTA EFIGENIA,...,1924,1,13.00,Loja,Comercial horizontal - padrão B,De esquina,0.2,1963,1,0
1,0010030002-2,2026,1,01/01/26,00-0,03812-1,R S CAETANO,19.0,NaN,SANTA EFIGENIA,...,1944,1,6.00,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
2,0010030003-0,2026,1,01/01/26,00-0,03812-1,R S CAETANO,27.0,NaN,SANTA EFIGENIA,...,1965,2,7.85,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
3,0010030004-9,2026,1,01/01/26,00-0,03812-1,R S CAETANO,33.0,NaN,NaN,...,1944,1,6.05,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
4,0010030005-7,2026,1,01/01/26,00-0,03812-1,R S CAETANO,39.0,NaN,NaN,...,2004,2,6.70,Loja,Comercial horizontal - padrão B,Normal,0.8,1963,1,0


In [4]:
from pathlib import Path

lotes_dir = Path('Data/lotes_shp')
shapefiles = sorted(lotes_dir.glob('*.shp'))

lotes_parts = []
for shp_path in shapefiles:
    gdf = gpd.read_file(shp_path)
    gdf['source_file'] = shp_path.name
    lotes_parts.append(gdf)

if lotes_parts:
    lotes = gpd.GeoDataFrame(pd.concat(lotes_parts, ignore_index=True), crs=lotes_parts[0].crs)
    print(f'Loaded {len(lotes)} features from {len(shapefiles)} shapefile(s).')
    print('CRS:', lotes.crs)
    display(lotes.head())
else:
    lotes = gpd.GeoDataFrame()
    print('No shapefiles found in', lotes_dir)

Loaded 1684088 features from 96 shapefile(s).
CRS: None


,lo_setor,lo_quadra,lo_lote,lo_condomi,lo_tp_quad,lo_tp_lote,geometry,source_file
0,052,078,0095,00,F,F,"POLYGON ((339749.727 7392905.356, 339752.346 7...",SIRGAS_SHP_LOTES_01_AGUA_RASA.shp
1,052,036,0088,00,F,F,"POLYGON ((338982.771 7393377.589, 338985.18 73...",SIRGAS_SHP_LOTES_01_AGUA_RASA.shp
2,052,367,0046,00,F,F,"POLYGON ((340096.834 7393138.43, 340096.602 73...",SIRGAS_SHP_LOTES_01_AGUA_RASA.shp
3,052,117,0028,00,F,F,"POLYGON ((339681.307 7392295.553, 339680.151 7...",SIRGAS_SHP_LOTES_01_AGUA_RASA.shp
4,052,254,0006,00,F,F,"POLYGON ((339473.181 7393288.725, 339476.427 7...",SIRGAS_SHP_LOTES_01_AGUA_RASA.shp


In [ ]:

import re
from typing import Any

# --- Passo 2: Criação da chave primária SQL_BASE para lotes ---
# Garantir que os campos numéricos sejam tratados como inteiros sem decimais.
def _format_key_part(series: pd.Series, width: int) -> pd.Series:
    return (
        series.astype(str)
        .str.replace(r'\D+', '', regex=True) # Remove letras preservando numerais, idêntico ao processo do IPTU
        .replace('', '0')
        .astype(int)
        .astype(str)
        .str.zfill(width)
    )
lotes = lotes.copy()
lotes['SQL_BASE'] = (
    _format_key_part(lotes['lo_setor'], 3)
    + _format_key_part(lotes['lo_quadra'], 3)
    + _format_key_part(lotes['lo_lote'], 4)
)

# --- Passo 3: Criação da chave primária SQL_BASE para IPTU ---
iptu = iptu_2026.copy()
iptu['NUMERO_DO_CONTRIBUINTE_CLEAN'] = (
    iptu['NUMERO DO CONTRIBUINTE']
    .astype(str)
    .str.replace(r'\D+', '', regex=True)
    .str.zfill(11)
)
iptu['SQL_BASE'] = iptu['NUMERO_DO_CONTRIBUINTE_CLEAN'].str.slice(0, 10)

# --- Passo 4: Agregação do IPTU ---
agg_columns: dict[str, Any] = {
    'VALOR DO M2 DO TERRENO': 'mean',
    'VALOR DO M2 DE CONSTRUCAO': 'mean',
    'ANO DA CONSTRUCAO CORRIGIDO': 'max',
    'TIPO DE PADRAO DA CONSTRUCAO': 'first',
    'AREA DO TERRENO': 'first',
}

for optional_column in [
    'TIPO DE IMOVEL',
    'LOGRADOURO',
    'CEP',
    'NOME DO CONTRIBUINTE',
]:
    if optional_column in iptu.columns:
        agg_columns[optional_column] = 'first'

iptu_agg = (
    iptu
    .groupby('SQL_BASE', as_index=False)
    .agg(agg_columns)
    .reset_index(drop=True)
)

# --- Passo 5: Consolidar lotes duplicados e fazer o merge ---
if lotes['SQL_BASE'].duplicated().any():
    print(f"Consolidando geometrias...")
    lotes = lotes.dissolve(by='SQL_BASE', aggfunc='first').reset_index()
merged = lotes.merge(
    iptu_agg,
    on='SQL_BASE',
    how='left',
    validate='1:1' # Regra rígida mantida!
)
# --- Passo 6: Limpeza e exportação ---
vars_alvo = ['VALOR DO M2 DO TERRENO', 'VALOR DO M2 DE CONSTRUCAO']
merged = merged.dropna(subset=vars_alvo).copy()
print(f'\nResultado da junção validada: {len(merged)} imóveis úteis.')
# CORREÇÃO CRÍTICA DO CRS: SIRGAS 2000 UTM Zone 23S (Usado na base do GeoSampa)
merged = merged.set_crs(epsg=31983, allow_override=True)
print(f'Projeção fixada em: {merged.crs}')
# CORREÇÃO FATOR DE DESEMPENHO: Exportar para .parquet
output_dir = Path('Data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'lotes_iptu_merged.parquet'
# Exportando Geometrias + DataFrame compactado e em formato colunar!
merged.to_parquet(output_path)
print(f"✅ Exportado focado em Alta Performance para: {output_path}")

Resultado da junção:
     SQL_BASE  VALOR DO M2 DO TERRENO  VALOR DO M2 DE CONSTRUCAO
5  0010030001                  4394.0                     2856.0
6  0010030002                  4394.0                     2856.0
7  0010030003                  4394.0                     2856.0
8  0010030004                  4394.0                     2856.0
9  0010030005                  4394.0                     2856.0

Info do GeoDataFrame unido:
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1611766 entries, 5 to 1648780
Data columns (total 14 columns):
 #   Column                        Non-Null Count    Dtype   
---  ------                        --------------    -----   
 0   SQL_BASE                      1611766 non-null  object  
 1   geometry                      1611766 non-null  geometry
 2   lo_setor                      1611766 non-null  object  
 3   lo_quadra                     1611766 non-null  object  
 4   lo_lote                       1611766 non-null  object  
 5   lo_co

c:\Users\damas\anaconda3\Lib\site-packages\pyogrio\geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


Exportado para: Data\processed\lotes_iptu_merged.geojson


In [5]:

from scipy.spatial import cKDTree

# ----------------------------------------------------------
# Carregamento da base principal
# ----------------------------------------------------------
lotes = gpd.read_parquet('Data/processed/lotes_iptu_merged.parquet')

# ----------------------------------------------------------
# Carregamento das camadas urbanas
# ----------------------------------------------------------
parques = gpd.read_file('Data/parques/cadparcs_parque_unidade_conservacao.shp')

metro = gpd.read_file('Data/transporte/estacao_metro_v2.shp')
trem = gpd.read_file('Data/transporte/estacao_trem_v2.shp')
onibus = gpd.read_file('Data/transporte/terminal_onibus_v2.shp')

# ----------------------------------------------------------
# Padronização do CRS
# ----------------------------------------------------------
CRS_PADRAO = 31983

for nome, gdf in [
    ("Parques", parques),
    ("Metrô", metro),
    ("Trem", trem),
    ("Ônibus", onibus)
]:
    if gdf.crs is None or gdf.crs.to_epsg() != CRS_PADRAO:
        print(f"Corrigindo CRS: {nome}")
        gdf.to_crs(epsg=CRS_PADRAO, inplace=True)

In [6]:
# ----------------------------------------------------------
# Centróides
# ----------------------------------------------------------
lotes_centroides = lotes.geometry.centroid
parques_centroides = parques.geometry.centroid

# ----------------------------------------------------------
# Função de distância mínima usando cKDTree
# ----------------------------------------------------------
def calcular_distancia_minima(pontos_origem, pontos_destino):
    origem_coords = np.array(list(zip(pontos_origem.x, pontos_origem.y)))
    destino_coords = np.array(list(zip(pontos_destino.x, pontos_destino.y)))

    arvore_destino = cKDTree(destino_coords)

    distancias, indices = arvore_destino.query(origem_coords, k=1)

    return distancias

# ----------------------------------------------------------
# Criação das features de distância
# ----------------------------------------------------------
lotes['DIST_METRO_M'] = calcular_distancia_minima(lotes_centroides, metro.geometry)
lotes['DIST_TREM_M'] = calcular_distancia_minima(lotes_centroides, trem.geometry)
lotes['DIST_ONIBUS_M'] = calcular_distancia_minima(lotes_centroides, onibus.geometry)
lotes['DIST_PARQUE_M'] = calcular_distancia_minima(lotes_centroides, parques_centroides)

# ----------------------------------------------------------
# Validação rápida
# ----------------------------------------------------------
colunas_novas = [
    'DIST_METRO_M',
    'DIST_TREM_M',
    'DIST_ONIBUS_M',
    'DIST_PARQUE_M'
]

print(lotes[colunas_novas].describe().round(1))

# ----------------------------------------------------------
# Exportação do dataset com features espaciais
# ----------------------------------------------------------
lotes.to_parquet('Data/processed/lotes_iptu_features.parquet')

print("✅ Arquivo salvo: Data/processed/lotes_iptu_features.parquet")

       DIST_METRO_M  DIST_TREM_M  DIST_ONIBUS_M  DIST_PARQUE_M
count     1611766.0    1611766.0      1611766.0      1611766.0
mean         3710.0       3429.7         2594.9         1166.2
std          3435.1       2102.6         1287.9          686.2
min             2.8          8.5            6.7            0.0
25%          1275.8       1703.8         1604.5          659.5
50%          2599.7       3096.1         2504.5         1043.1
75%          5047.9       4904.3         3458.4         1535.4
max         28681.8      17269.1        10863.3         4437.2
✅ Arquivo salvo: Data/processed/lotes_iptu_features.parquet


In [7]:


# ----------------------------------------------------------
# Carregamento da base com features
# ----------------------------------------------------------
lotes = gpd.read_parquet('Data/processed/lotes_iptu_features.parquet')

# ----------------------------------------------------------
# Carregamento do zoneamento
# ----------------------------------------------------------
zona = gpd.read_file('Data/zoneamento/perimetro_zona_lei_18177_24.shp')

# ----------------------------------------------------------
# Padronização do CRS
# ----------------------------------------------------------
CRS_PADRAO = 31983

if zona.crs is None or zona.crs.to_epsg() != CRS_PADRAO:
    zona.to_crs(epsg=CRS_PADRAO, inplace=True)

In [8]:
# ----------------------------------------------------------
# Spatial Join usando centróides dos lotes
# ----------------------------------------------------------
lotes_centroides = lotes.copy()
lotes_centroides['geometry'] = lotes.geometry.centroid

lotes_com_zona = gpd.sjoin(
    lotes_centroides,
    zona,
    how='left',
    predicate='intersects'
)

# ----------------------------------------------------------
# Correção para lotes duplicados no sjoin
# Alguns centróides podem cair em fronteiras de zonas
# ----------------------------------------------------------
lotes_com_zona = lotes_com_zona[
    ~lotes_com_zona.index.duplicated(keep='first')
]

# ----------------------------------------------------------
# Coluna correta do shapefile de zoneamento
# ----------------------------------------------------------
coluna_sigla = 'tx_zoneame'

lotes['ZONEAMENTO'] = lotes_com_zona[coluna_sigla].fillna('Sem Zona')

# ----------------------------------------------------------
# Exportação do dataset master
# ----------------------------------------------------------
lotes.to_parquet('Data/processed/lotes_iptu_master.parquet')

print("✅ Arquivo salvo: Data/processed/lotes_iptu_master.parquet")

✅ Arquivo salvo: Data/processed/lotes_iptu_master.parquet
